dataset prep

In [ ]:
import platform

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm

Device selection: `GPU` if available, else `CPU`.

In [ ]:
def get_device() -> torch.device:
    """Select the best available torch device across CUDA, Apple MPS, and CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    # MPS = Apple's Metal backend (M-series). Guard both attr + availability
    # so older torch builds without MPS don't blow up.
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()

print(f"OS:\t{platform.system()} ({platform.machine()})")
print(f"Device:\t{device}")

match device.type:
    case "cuda":
        print(f"GPU:\t{torch.cuda.get_device_name(0)}")
    case "mps":
        print("GPU:\tApple Metal (MPS)")
    case _:
        print("GPU:\tnone — running on CPU")

Defining hyperparameters (adjus as needed ; maybe a search ?)

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 4  # Set to 0 on Windows if you get errors
LEARNING_RATE = 1e-3
NUM_EPOCHS = 10

Data loading MNIST dataset. We first define a transform to convert PIL images to torch tensors with values in range [0,1]

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),  # Converts PIL image to torch tensor in [0, 1]
])

Downloading train and test datasets and streamlining into `DataLoader` classes (why is it convenient ; helper functions ?; iterators ?)

In [ ]:
train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    transform=transform,
    download=True
)

test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    transform=transform,
    download=True
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

print(f"Train set: {len(train_dataset)} images")
print(f"Test set: {len(test_dataset)} images")
print(f"Image shape: {train_dataset[0][0].shape}")  # Should be (1, 28, 28)

Visualization helper functions:

In [ ]:
def visualize_batch(images, labels=None, title="Batch", nrows=4, ncols=8):
    """
    Display a batch of MNIST images.
    
    Args:
        images: tensor of shape (B, 1, 28, 28) or (B, 28, 28)
        labels: optional tensor of digit labels (B,)
        title: title for the figure
        nrows, ncols: grid layout
    """
    # Ensure 4D: (B, C, H, W)
    if images.dim() == 3:
        images = images.unsqueeze(1)
    
    # Move to CPU for plotting
    images = images.detach().cpu()
    if labels is not None:
        labels = labels.cpu().numpy()
    
    _, axes = plt.subplots(nrows, ncols, figsize=(12, 6))
    axes = axes.flatten()
    
    n_show = min(len(images), nrows * ncols)
    for i in range(n_show):
        ax = axes[i]
        img = images[i].squeeze()  # Remove channel dim for display
        ax.imshow(img, cmap='gray')
        ax.axis('off')
        if labels is not None:
            ax.set_title(f"Label: {labels[i]}", fontsize=8)
    
    # Hide unused subplots
    for i in range(n_show, len(axes)):
        axes[i].axis('off')
    
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


def visualize_reconstructions(originals, reconstructions, labels=None, nrows=4, ncols=4):
    """
    Side-by-side comparison of originals and reconstructions.
    
    Args:
        originals: (B, 1, 28, 28) or (B, 28, 28)
        reconstructions: same shape
        labels: optional labels
    """
    # Ensure 4D
    if originals.dim() == 3:
        originals = originals.unsqueeze(1)
    if reconstructions.dim() == 3:
        reconstructions = reconstructions.unsqueeze(1)
    
    originals = originals.detach().cpu()
    reconstructions = reconstructions.detach().cpu()
    if labels is not None:
        labels = labels.cpu().numpy()
    
    _, axes = plt.subplots(nrows, ncols * 2, figsize=(12, nrows * 1.5))
    axes = axes.flatten()
    
    n_show = min(len(originals), nrows * ncols)
    for i in range(n_show):
        # Original
        ax_orig = axes[2 * i]
        img_orig = originals[i].squeeze()
        ax_orig.imshow(img_orig, cmap='gray')
        ax_orig.axis('off')
        if labels is not None:
            ax_orig.set_title(f"Orig: {labels[i]}", fontsize=8)
        else:
            ax_orig.set_title("Original", fontsize=8)
        
        # Reconstruction
        ax_recon = axes[2 * i + 1]
        img_recon = reconstructions[i].squeeze()
        ax_recon.imshow(img_recon, cmap='gray')
        ax_recon.axis('off')
        ax_recon.set_title("Reconstruction", fontsize=8)
    
    plt.tight_layout()
    plt.show()


def plot_latent_2d(latents, labels, title="2D Latent Space"):
    """
    Scatter plot of 2D latent codes colored by digit label.
    
    Args:
        latents: tensor of shape (N, 2)
        labels: tensor of shape (N,) with digit labels
    """
    latents = latents.detach().cpu().numpy()
    labels = labels.cpu().numpy()
    
    _, ax = plt.subplots(figsize=(10, 8))
    
    colors = plt.cm.tab10(np.arange(10))
    for digit in range(10):
        mask = labels == digit
        ax.scatter(
            latents[mask, 0],
            latents[mask, 1],
            c=[colors[digit]],
            label=f"Digit {digit}",
            s=20,
            alpha=0.6
        )
    
    ax.set_xlabel("Latent Dim 1")
    ax.set_ylabel("Latent Dim 2")
    ax.set_title(title)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_interpolation(decoder, z1, z2, nsteps=10):
    """
    Decode points along a linear interpolation from z1 to z2.
    
    Args:
        decoder: a module that maps latents to images
        z1, z2: latent codes (D,) or (1, D) tensors
        nsteps: number of interpolation points
    """
    # Ensure (1, D) shape
    if z1.dim() == 1:
        z1 = z1.unsqueeze(0)
    if z2.dim() == 1:
        z2 = z2.unsqueeze(0)
    
    decoder = decoder.to(device).eval()
    z1 = z1.to(device)
    z2 = z2.to(device)
    
    alphas = np.linspace(0, 1, nsteps)
    with torch.no_grad():
        z_interp = torch.stack([
            (1 - alpha) * z1 + alpha * z2
            for alpha in alphas
        ]).squeeze(1)  # (nsteps, D)
        images = decoder(z_interp)
    
    _, axes = plt.subplots(1, nsteps, figsize=(nsteps * 1.5, 2))
    for i, img in enumerate(images):
        ax = axes[i]
        ax.imshow(img.squeeze().cpu(), cmap='gray')
        ax.axis('off')
        ax.set_title(f"α={alphas[i]:.2f}", fontsize=8)
    
    plt.suptitle("Interpolation in Latent Space")
    plt.tight_layout()
    plt.show()

Grab one batch and visualize:

In [ ]:
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}")
print(f"Labels: {labels[:8]}")

visualize_batch(images, labels, title="Sample MNIST Batch", nrows=4, ncols=8)

Defining the autoencoder model architecture:

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_channels=1, latent_dim=32):
        super().__init__()
        self.latent_dim = latent_dim
        self.conv1 = nn.Conv2d(input_channels, 32, kernel_size=4, stride=2, padding=1) # 1x28x28 -> 32x14x14
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1) # 32x14x14 -> 64x7x7
        self.fc_out = nn.Linear(64 * 7 * 7, latent_dim) # 3136 -> latent_dim
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.view(x.size(0), -1)  # Flatten
        z = self.fc_out(x)
        return z
    
class Decoder(nn.Module):
    def __init__(self, output_channels=1, latent_dim=32):
        super().__init__()
        self.latent_dim = latent_dim
        self.fc_in = nn.Linear(latent_dim, 64 * 7 * 7) # latent_dim -> 3136
        self.deConv1 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1) # 64x7x7 -> 32x14x14
        self.deConv2 = nn.ConvTranspose2d(32, output_channels, kernel_size=4, stride=2, padding=1) # 32x14x14 -> 1x28x28
    
    def forward(self, z):
        x = F.relu(self.fc_in(z))
        x = x.view(-1, 64, 7, 7)  # Reshape to spatial
        x = F.relu(self.deConv1(x))
        x = torch.sigmoid(self.deConv2(x))  # Pixel values in [0, 1]
        return x

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, input_channels=1, output_channels=1, latent_dim=32):
        super().__init__()
        self.encoder = Encoder(input_channels=input_channels, latent_dim=latent_dim)
        self.decoder = Decoder(output_channels=output_channels, latent_dim=latent_dim)
        
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

In [ ]:
def reconstruction_loss_mse(x, x_recon):
    """MSE reconstruction loss."""
    return F.mse_loss(x, x_recon)

def reconstruction_loss_bce(x, x_recon):
    """Binary cross-entropy per pixel."""
    return F.binary_cross_entropy(x_recon, x)

def kl_divergence_normal(mu, log_var):
    """KL divergence for isotropic Gaussian against standard normal."""
    # KL = -0.5 * sum(1 + log_var - mu^2 - exp(log_var))
    kl = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    return kl / x.size(0)  # Average over batch

In [ ]:
def train_epoch(model, train_loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images = images.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = loss_fn(images, outputs)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)

def evaluate(model, test_loader, loss_fn, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            loss = loss_fn(images, outputs)
            total_loss += loss.item()
    
    return total_loss / len(test_loader)

In [ ]:
model = AutoEncoder(
    input_channels=1, output_channels=1, latent_dim=32
)
model.to(device)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
train_epoch(
    model=model, train_loader=train_loader, optimizer=optimizer, loss_fn=reconstruction_loss_mse, device=device
)

In [ ]:
for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    n_batches = 0

    # train_loader has shuffle=True, so batches are freshly reshuffled
    # every epoch -> this is mini-batch stochastic gradient descent.
    progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}", leave=True)
    for images, _ in progress:
        images = images.to(device)

        # Forward pass
        outputs = model(images)
        loss = reconstruction_loss_mse(images, outputs)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        n_batches += 1
        progress.set_postfix(loss=f"{running_loss / n_batches:.4f}")

    print(f"Epoch {epoch + 1}/{NUM_EPOCHS} — avg loss: {running_loss / n_batches:.4f}")